In [1]:
"""
=============================================================================
Breast Cancer Detection — Ultrasound Module (Local Machine)
Model   : MobileNetV3-Large (Transfer Learning)
Dataset : BUSI (Breast Ultrasound Images Dataset)
Output  : INT8 TFLite quantized model
=============================================================================
"""

import os
import glob
import random
import hashlib
import time
import warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks, mixed_precision
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")



I0000 00:00:1776095931.040634    2519 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776095932.295631    2519 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776095936.363634    2519 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# =============================================================================
# 1. CONFIGURATION
# =============================================================================

if os.name == "nt":
    DATASET_ROOT = r"C:\Users\mmando\Downloads\Dataset_BUSI_with_GT"
    OUTPUT_DIR   = r"C:\Users\mmando\Documents\School\ML\Output_Ultrasound"
else:
    DATASET_ROOT = os.path.expanduser("~/Dataset_BUSI_with_GT")
    # If the folder is named differently in WSL, adjust here:
    if not os.path.exists(DATASET_ROOT):
        DATASET_ROOT = os.path.expanduser("~/BUSI") 
    OUTPUT_DIR   = os.path.expanduser("~/ultrasound_output")

IMG_SIZE        = 224
BATCH_SIZE      = 16       # Smaller batch size for the smaller dataset
EPOCHS_FROZEN   = 100
EPOCHS_FINETUNE = 50       # Kept short to prevent overfitting
LEARNING_RATE   = 0.0001
DROPOUT_RATE    = 0.5      # Increased for heavy regularization
L2_DECAY        = 0.0005   # Increased L2 penalty
PATIENCE_STOP   = 15
PATIENCE_LR     = 8
SEED            = 42

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
##mixed_precision.set_global_policy("mixed_float16")

os.makedirs(OUTPUT_DIR, exist_ok=True)
CACHE_DIR          = os.path.join(OUTPUT_DIR, "cache")
CHECKPOINT_PATH_P1 = os.path.join(OUTPUT_DIR, "best_busi_phase1.keras")
CHECKPOINT_PATH_P2 = os.path.join(OUTPUT_DIR, "best_busi_phase2.keras")
TFLITE_PATH        = os.path.join(OUTPUT_DIR, "ultrasound_mobilenetv3_int8.tflite")
PLOT_DIR           = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(PLOT_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)


# =============================================================================
# 2. DATA LOADING (BUSI Specific - FIXED PATH MATCHING)
# =============================================================================

def load_dataset() -> pd.DataFrame:
    print(f"Scanning {DATASET_ROOT}...")
    
    # Grab all image files (checks multiple extensions just in case)
    all_files = []
    for ext in ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG'):
        all_files.extend(glob.glob(os.path.join(DATASET_ROOT, "**", ext), recursive=True))
    
    # Filter out ground truth masks
    img_paths = [p for p in all_files if "mask" not in os.path.basename(p).lower()]
    
    data = []
    for path in img_paths:
        # Convert the entire absolute path to lowercase for bulletproof matching
        path_lower = path.lower()
        
        # Binary Classification: Normal/Benign (0) vs Malignant (1)
        if "malignant" in path_lower:
            label = 1
            orig_class = "malignant"
        elif "benign" in path_lower:
            label = 0
            orig_class = "benign"
        elif "normal" in path_lower:
            label = 0
            orig_class = "normal"
        else:
            continue
            
        data.append({"img_path": path, "label": label, "original_class": orig_class})
        
    df = pd.DataFrame(data)
    
    print(f"\n  Total Images : {len(df)}")
    if not df.empty:
        print(f"  Non-Malignant: {(df.label==0).sum()} (Normal + Benign)")
        print(f"  Malignant    : {(df.label==1).sum()}")
    
    # Failsafes
    if len(df) == 0:
        raise FileNotFoundError(f"No valid images found inside {DATASET_ROOT}.")
    if (df.label==1).sum() == 0:
        raise ValueError("CRITICAL ERROR: 0 Malignant images found. Check your DATASET_ROOT and folder structure.")
        
    return df


# =============================================================================
# 3. PREPROCESSING — Speckle Reduction
# =============================================================================

def apply_speckle_reduction(img_bgr: np.ndarray) -> np.ndarray:
    """
    Median blur is highly effective at removing salt-and-pepper / speckle noise
    in ultrasound without heavily blurring the structural edges of a mass.
    """
    # 3x3 kernel size
    return cv2.medianBlur(img_bgr, 3)

def _cache_path(img_path: str) -> str:
    key = hashlib.md5(img_path.encode()).hexdigest()
    return os.path.join(CACHE_DIR, f"{key}.npy")

def build_cache(paths: list) -> None:
    missing = [p for p in paths if not os.path.exists(_cache_path(p))]
    if not missing:
        print(f"  Cache up to date — {len(paths)} files already cached.")
        return
    
    for p in tqdm(missing, desc="  Building cache"):
        img = cv2.imread(p)
        if img is not None:
            img = apply_speckle_reduction(img)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            arr = img.astype(np.float32) / 255.0
            np.save(_cache_path(p), arr)


def load_and_preprocess(img_path: str, augment: bool = False) -> np.ndarray:
    cp = _cache_path(img_path)
    if os.path.exists(cp):
        img_float = np.load(cp)
        img       = (img_float * 255).astype(np.uint8)
        img       = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    else:
        img = cv2.imread(img_path)
        img = apply_speckle_reduction(img)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    if augment:
        if random.random() < 0.5:
            img = cv2.flip(img, 1)
        # Slightly more aggressive rotation for small dataset
        angle = random.uniform(-25, 25)
        M     = cv2.getRotationMatrix2D((IMG_SIZE // 2, IMG_SIZE // 2), angle, 1)
        img   = cv2.warpAffine(img, M, (IMG_SIZE, IMG_SIZE))

        # Brightness
        hsv          = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 2] = np.clip(hsv[:, :, 2] * random.uniform(0.7, 1.3), 0, 255)
        img          = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0


# =============================================================================
# 4. tf.data PIPELINE
# =============================================================================

def make_tf_dataset(paths: list, labels: list, augment: bool = False, shuffle: bool = False):
    def _load(path, label):
        img = tf.numpy_function(
            func=lambda p: load_and_preprocess(p.decode(), augment=augment),
            inp=[path], Tout=tf.float32
        )
        img.set_shape([IMG_SIZE, IMG_SIZE, 3])
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, np.array(labels, dtype=np.float32)))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(_load, num_parallel_calls=8)
    if not augment:
        ds = ds.cache()
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


    


# =============================================================================
# 5. MODEL & TRAINING 
# =============================================================================

def build_model(freeze_base: bool = True):
    base = MobileNetV3Large(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet", include_preprocessing=False
    )
    base.trainable = not freeze_base

    reg = tf.keras.regularizers.l2(L2_DECAY)
    inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = base(inp, training=not freeze_base)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dropout(DROPOUT_RATE)(x)
    x   = layers.Dense(128, activation="relu", kernel_regularizer=reg)(x) # Reduced width
    x   = layers.Dropout(DROPOUT_RATE)(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    return Model(inp, out, name="MobileNetV3_Ultrasound"), base

def compile_model(model: Model) -> None:
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy", 
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.AUC(curve="PR", name="prc")]
    )

def train(train_ds, val_ds, cw_dict: dict):
    model, base = build_model(freeze_base=True)
    compile_model(model)

    print("\n─── Phase 1: Frozen base ───")
    cbs_p1 = [
        callbacks.ModelCheckpoint(CHECKPOINT_PATH_P1, monitor="val_auc", mode="max", save_best_only=True, verbose=1),
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=PATIENCE_STOP, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=PATIENCE_LR, min_lr=1e-7)
    ]
    h_frozen = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FROZEN, class_weight=cw_dict, callbacks=cbs_p1, verbose=1)
    model.load_weights(CHECKPOINT_PATH_P1)

    print("\n─── Phase 2: Fine-tuning top 10 layers ───")
    base.trainable = True
    for layer in base.layers[:-10]: # Very strict unfreezing for small dataset
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE * 0.1),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.AUC(curve="PR", name="prc")]
    )
    
    cbs_p2 = [
        callbacks.ModelCheckpoint(CHECKPOINT_PATH_P2, monitor="val_auc", mode="max", save_best_only=True, verbose=1),
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=PATIENCE_STOP, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=PATIENCE_LR, min_lr=1e-7)
    ]
    h_finetune = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE, class_weight=cw_dict, callbacks=cbs_p2, verbose=1)
    model.load_weights(CHECKPOINT_PATH_P2)

    return model

# =============================================================================
# 6. GRAD-CAM 
# =============================================================================

def make_gradcam(img_array: np.ndarray, model: Model) -> np.ndarray:
    inp = tf.cast(img_array[np.newaxis], tf.float32)
    base_model, head_layers, found_base = None, [], False

    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            base_model = layer
            found_base = True
        elif found_base:
            head_layers.append(layer)

    with tf.GradientTape() as tape:
        conv_out = base_model(inp, training=False)
        tape.watch(conv_out)
        x = conv_out
        for layer in head_layers:
            x = layer(x, training=False) if isinstance(layer, tf.keras.layers.Dropout) else layer(x)
        class_score = tf.cast(x[:, 0], tf.float32)

    grads = tf.cast(tape.gradient(class_score, conv_out), tf.float32)
    conv_out = tf.cast(conv_out, tf.float32)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0)
    return (heatmap / (tf.math.reduce_max(heatmap) + 1e-8)).numpy()

def save_gradcam_grid(model: Model, X_te: list, y_te: list) -> None:
    b_idx = [i for i, l in enumerate(y_te) if l == 0][:2]
    m_idx = [i for i, l in enumerate(y_te) if l == 1][:2]
    sample_idx = b_idx + m_idx

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for col, idx in enumerate(sample_idx):
        img = load_and_preprocess(X_te[idx], augment=False)
        prob = float(model.predict(img[np.newaxis], verbose=0)[0, 0])
        label = "Malignant" if y_te[idx] == 1 else "Non-Malignant"
        
        hm = make_gradcam(img, model)
        h = cv2.applyColorMap(np.uint8(255 * cv2.resize(hm, (IMG_SIZE, IMG_SIZE))), cv2.COLORMAP_JET)
        h = cv2.cvtColor(h, cv2.COLOR_BGR2RGB)
        ov = np.uint8(img * 255 * 0.6 + h * 0.4)

        axes[0, col].imshow(img)
        axes[0, col].set_title(f"Original ({label})", fontsize=10)
        axes[0, col].axis("off")
        axes[1, col].imshow(ov)
        axes[1, col].set_title(f"P(malignant)={prob:.2f}", fontsize=10)
        axes[1, col].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "gradcam.png"), dpi=150)
    plt.close()

# =============================================================================
# 7. MAIN PIPELINE
# =============================================================================

def main():
    print("─── Step 1: Loading BUSI Dataset ───")
    df = load_dataset()
    paths, labels = df["img_path"].tolist(), df["label"].tolist()

    X_tr, X_tmp, y_tr, y_tmp = train_test_split(paths, labels, test_size=1-TRAIN_RATIO, stratify=labels, random_state=SEED)
    X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=TEST_RATIO/(VAL_RATIO+TEST_RATIO), stratify=y_tmp, random_state=SEED)
    
    cw = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
    cw_dict = {i: float(w) for i, w in enumerate(cw)}

    print("\n─── Step 2: Caching & Data Pipeline ───")
    build_cache(paths)
    train_ds = make_tf_dataset(X_tr, y_tr, augment=True, shuffle=True)
    val_ds   = make_tf_dataset(X_val, y_val, augment=False, shuffle=False)
    test_ds  = make_tf_dataset(X_te, y_te, augment=False, shuffle=False)

    print("\n─── Step 3: Training ───")
    model = train(train_ds, val_ds, cw_dict)

    print("\n─── Step 4: Evaluation ───")
    y_prob = model.predict(test_ds).ravel()
    print(f"  ROC-AUC : {roc_auc_score(y_te, y_prob):.4f}")
    save_gradcam_grid(model, X_te, y_te)

    print("\n─── Step 5: TFLite Quantization ───")
    cal_imgs = np.array([load_and_preprocess(p) for p in X_tr[:500]], dtype=np.float32)
    def rep_data():
        for img in cal_imgs: yield [img[np.newaxis]]
        
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = rep_data
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    
    try:
        tflite_model = converter.convert()
        with open(TFLITE_PATH, "wb") as f:
            f.write(tflite_model)
        print(f"  Saved INT8 Model → {TFLITE_PATH}")
    except Exception as e:
        print(f"  INT8 failed ({e}) — falling back to float16")
        conv2 = tf.lite.TFLiteConverter.from_keras_model(model)
        conv2.optimizations = [tf.lite.Optimize.DEFAULT]
        conv2.target_spec.supported_types = [tf.float16]
        tflite_model = conv2.convert()
        with open(TFLITE_PATH, "wb") as f:
            f.write(tflite_model)
        print(f"  Saved float16 Model → {TFLITE_PATH}")

if __name__ == "__main__":
    main()

─── Step 1: Loading BUSI Dataset ───
Scanning /home/mmando/BUSI...

  Total Images : 780
  Non-Malignant: 570 (Normal + Benign)
  Malignant    : 210

─── Step 2: Caching & Data Pipeline ───
  Cache up to date — 780 files already cached.


I0000 00:00:1776095940.806877    2519 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9694 MB memory:  -> device: 0, name: NVIDIA RTX A3000 12GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6



─── Step 3: Training ───

─── Phase 1: Frozen base ───
Epoch 1/100


I0000 00:00:1776095952.887923    2680 service.cc:153] XLA service 0x71bf1c061770 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776095952.887976    2680 service.cc:161]   StreamExecutor [0]: NVIDIA RTX A3000 12GB Laptop GPU, Compute Capability 8.6 (Driver: 12.8.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.20.0)
I0000 00:00:1776095953.163910    2680 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776095955.078218    2680 cuda_dnn.cc:461] Loaded cuDNN version 92000
I0000 00:00:1776095955.241161    2680 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_10747__.180
E0000 00:00:1776095959.566366    2680 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1776095970.874639    2680 cuda_timer.cc:87] Delay kernel timed 

34/35 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.5907 - auc: 0.5267 - loss: 1.0415 - prc: 0.2696

...

INFO:tensorflow:Assets written to: /tmp/tmprj2u_2v3/assets


Saved artifact at '/tmp/tmprj2u_2v3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_202')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  125071789635344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125071789635728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125071789637072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125071789636688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125071789635152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125071789635536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125071789636496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125071789636304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125071789635920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125071789634960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1250717896

W0000 00:00:1776096513.925689    2519 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776096513.925756    2519 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1776096513.926240    2519 reader.cc:83] Reading SavedModel from: /tmp/tmprj2u_2v3
I0000 00:00:1776096513.937114    2519 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1776096513.937147    2519 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmprj2u_2v3
I0000 00:00:1776096514.042589    2519 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1776096514.066893    2519 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1776096514.902367    2519 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmprj2u_2v3
I0000 00:00:1776096515.079382    2519 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 1153158 microseconds.


  Saved INT8 Model → /home/mmando/ultrasound_output/ultrasound_mobilenetv3_int8.tflite


fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8
W0000 00:00:1776096573.153636    2519 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.


In [4]:
import subprocess, sys, os

env = {
    **os.environ,
    "CUDA_VISIBLE_DEVICES":  "",
    "TF_CPP_MIN_LOG_LEVEL":  "3",
    "TF_ENABLE_ONEDNN_OPTS": "0",
    "OMP_NUM_THREADS":       "1",
}

# Step 0: pin protobuf to a version compatible with TF 2.16/2.17
subprocess.run(
    [sys.executable, "-m", "pip", "install", "protobuf==3.20.3", "-q"],
    env=env, check=True
)
print("protobuf pinned ✓")

# Step 1: run conversion
result = subprocess.run(
    [sys.executable, "/home/mmando/convert_via_onnx.py"],
    capture_output=True, text=True, env=env
)

print(result.stdout)
if result.returncode != 0:
    print("=== STDERR ===")
    print(result.stderr[-4000:])

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx2tf 2.4.0 requires protobuf==4.25.5, but you have protobuf 3.20.3 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.1 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.


protobuf pinned ✓
=== Step 1: checking dependencies ===
  installing tf2onnx ...

=== STDERR ===
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx2tf 2.4.0 requires protobuf==4.25.5, but you have protobuf 7.34.1 which is incompatible.
free(): double free detected in tcache 2

